# RQ3.2 — Class-level Analysis

This notebook reproduces the descriptive class-level comparison between the **zero-shot baseline** (`p0_baseline`) and the final **code-disambiguation** prompt (`p_code_disambiguation`).

The analysis is intentionally separate from the project-level repeated-measures analyses used in the four RQ3.2 prompt-refinement factors.

Here, precision, recall, and F1 are computed from predictions **pooled across all commits**. These results are diagnostic and are therefore not directly comparable to the mean project-level Macro-F1 values used in the inferential analyses.

Expected replication-package layout:

```text
results/rq3_2/
├── analysis/
│   └── rq3_2_class_analysis.ipynb
├── metrics/
│   └── class_metrics_baseline_vs_code_disambiguation.csv
├── predictions/
│   └── predictions.csv.gz
└── figures/
    ├── rq3_class_f1.pdf
    └── rq3_class_f1.png
```

The notebook recomputes the class-level metrics directly from `predictions.csv.gz`. If the metrics CSV already exists, it validates the recomputed values against it. Otherwise, it creates the CSV.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import precision_recall_fscore_support

EXPECTED_COMMITS = 7937
BASELINE_PROMPT = "p0_baseline"
FINAL_PROMPT = "p_code_disambiguation"

CLASSES = [
    "fix",
    "feat",
    "build",
    "ci",
    "docs",
    "style",
    "refactor",
    "perf",
    "test",
    "chore",
    "others",
]

PROMPT_LABELS = {
    BASELINE_PROMPT: "Baseline",
    FINAL_PROMPT: "Code disambiguation",
}

# Optional override. Set this to results/rq3_2 if automatic path resolution
# does not find the package files.
RQ3_2_ROOT_OVERRIDE = None


def candidate_roots():
    roots = []

    if RQ3_2_ROOT_OVERRIDE is not None:
        roots.append(Path(RQ3_2_ROOT_OVERRIDE).expanduser().resolve())

    cwd = Path.cwd().resolve()

    # Common launch locations:
    #   results/rq3_2/analysis/
    #   results/rq3_2/
    #   results/
    #   repository root
    roots.append(cwd)
    roots.extend(list(cwd.parents)[:5])

    unique = []
    seen = set()
    for root in roots:
        marker = str(root)
        if marker not in seen:
            unique.append(root)
            seen.add(marker)
    return unique


def resolve_rq3_2_root():
    candidates = []

    for root in candidate_roots():
        candidates.extend([
            root,
            root / "rq3_2",
            root / "results" / "rq3_2",
        ])

    unique = []
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        marker = str(candidate)
        if marker in seen:
            continue
        seen.add(marker)
        unique.append(candidate)

        if (
            (candidate / "predictions" / "predictions.csv.gz").is_file()
            and (candidate / "metrics").is_dir()
        ):
            return candidate

    attempted = "\n".join(f"  - {path}" for path in unique)
    raise FileNotFoundError(
        "Could not locate results/rq3_2.\n"
        f"Checked:\n{attempted}\n\n"
        "Set RQ3_2_ROOT_OVERRIDE explicitly if your package uses a "
        "different location."
    )


RQ3_2_ROOT = resolve_rq3_2_root()
PREDICTIONS_PATH = RQ3_2_ROOT / "predictions" / "predictions.csv.gz"
METRICS_PATH = (
    RQ3_2_ROOT
    / "metrics"
    / "class_metrics_baseline_vs_code_disambiguation.csv"
)
FIGURES_DIR = RQ3_2_ROOT / "figures"
FIGURE_PATH = FIGURES_DIR / "rq3_class_f1.png"
FIGURE_PDF_PATH = FIGURES_DIR / "rq3_class_f1.pdf"

print(f"RQ3.2 root : {RQ3_2_ROOT}")
print(f"Predictions: {PREDICTIONS_PATH}")
print(f"Metrics    : {METRICS_PATH}")
print(f"Figure (PNG): {FIGURE_PATH}")
print(f"Figure (PDF): {FIGURE_PDF_PATH}")


## 1. Load and validate the final predictions

Only the two treatments used in the class-level comparison are retained. The notebook verifies that both contain the full 7,937-commit cohort and that their targets refer to the same commits.


In [2]:
predictions = pd.read_csv(
    PREDICTIONS_PATH,
    compression="gzip",
    dtype={
        "prompt": "string",
        "project": "string",
        "commit_hash": "string",
        "target": "string",
        "prediction": "string",
    },
)

required_columns = {
    "prompt",
    "project",
    "commit_hash",
    "target",
    "prediction",
}
missing = sorted(required_columns - set(predictions.columns))
if missing:
    raise ValueError(
        f"Missing required columns in predictions.csv.gz: {missing}"
    )

comparison = predictions.loc[
    predictions["prompt"].isin([BASELINE_PROMPT, FINAL_PROMPT]),
    ["prompt", "project", "commit_hash", "target", "prediction"],
].copy()

if len(comparison) != EXPECTED_COMMITS * 2:
    raise ValueError(
        f"Expected {EXPECTED_COMMITS * 2:,} rows for the two prompts, "
        f"found {len(comparison):,}."
    )

counts = (
    comparison.groupby("prompt")
    .agg(
        rows=("commit_hash", "size"),
        projects=("project", "nunique"),
        commits=("commit_hash", "nunique"),
    )
)

display(counts)

for prompt in [BASELINE_PROMPT, FINAL_PROMPT]:
    if counts.loc[prompt, "rows"] != EXPECTED_COMMITS:
        raise ValueError(
            f"{prompt}: expected {EXPECTED_COMMITS:,} rows, "
            f"found {counts.loc[prompt, 'rows']:,}."
        )

duplicate_keys = comparison.duplicated(
    ["prompt", "project", "commit_hash"],
    keep=False,
)
if duplicate_keys.any():
    raise ValueError(
        "Duplicate prompt/project/commit_hash rows were found."
    )

baseline_keys = (
    comparison.loc[
        comparison["prompt"] == BASELINE_PROMPT,
        ["project", "commit_hash", "target"],
    ]
    .sort_values(["project", "commit_hash"])
    .reset_index(drop=True)
)

final_keys = (
    comparison.loc[
        comparison["prompt"] == FINAL_PROMPT,
        ["project", "commit_hash", "target"],
    ]
    .sort_values(["project", "commit_hash"])
    .reset_index(drop=True)
)

if not baseline_keys.equals(final_keys):
    raise ValueError(
        "Baseline and code-disambiguation do not contain the same "
        "project/commit/target cohort."
    )

unknown_targets = sorted(
    set(comparison["target"].dropna()) - set(CLASSES)
)
if unknown_targets:
    raise ValueError(
        f"Unexpected ground-truth classes: {unknown_targets}"
    )

print("Cohort validation: OK")


,rows,projects,commits
prompt,,,
p0_baseline,7937,30,7937
p_code_disambiguation,7937,30,7937


Cohort validation: OK


## 2. Recompute pooled class-level precision, recall, F1, and support

Predictions outside the 11-class taxonomy, such as strict invalid outputs or final abstentions, remain incorrect predictions. They are **not mapped to `others`** and are **not introduced as semantic classes** in the metric table.

By explicitly passing the 11 evaluated classes to `precision_recall_fscore_support`, such outputs still contribute false negatives to their ground-truth classes while remaining outside the semantic label set.


In [3]:
def compute_class_metrics(
    frame: pd.DataFrame,
    prompt: str,
) -> pd.DataFrame:
    subset = frame.loc[frame["prompt"] == prompt].copy()

    precision, recall, f1, support = precision_recall_fscore_support(
        subset["target"],
        subset["prediction"],
        labels=CLASSES,
        average=None,
        zero_division=0,
    )

    return pd.DataFrame({
        "policy": "strict_taxonomy",
        "prompt": prompt,
        "prompt_label": PROMPT_LABELS[prompt],
        "class": CLASSES,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support.astype(int),
    })


baseline_metrics = compute_class_metrics(
    comparison,
    BASELINE_PROMPT,
)
final_metrics = compute_class_metrics(
    comparison,
    FINAL_PROMPT,
)

if not np.array_equal(
    baseline_metrics["support"].to_numpy(),
    final_metrics["support"].to_numpy(),
):
    raise RuntimeError(
        "Class support differs between the two prompt treatments."
    )

delta_metrics = pd.DataFrame({
    "policy": "strict_taxonomy",
    "prompt": "delta_code_disambiguation_minus_baseline",
    "prompt_label": "Code disambiguation − Baseline",
    "class": CLASSES,
    "precision": (
        final_metrics["precision"].to_numpy()
        - baseline_metrics["precision"].to_numpy()
    ),
    "recall": (
        final_metrics["recall"].to_numpy()
        - baseline_metrics["recall"].to_numpy()
    ),
    "f1": (
        final_metrics["f1"].to_numpy()
        - baseline_metrics["f1"].to_numpy()
    ),
    "support": baseline_metrics["support"].to_numpy(),
})

computed_metrics = pd.concat(
    [baseline_metrics, final_metrics, delta_metrics],
    ignore_index=True,
)

display(
    computed_metrics.style.format({
        "precision": "{:.6f}",
        "recall": "{:.6f}",
        "f1": "{:.6f}",
    })
)


,policy,prompt,prompt_label,class,precision,recall,f1,support
0,strict_taxonomy,p0_baseline,Baseline,fix,0.693742,0.495247,0.577926,2104
1,strict_taxonomy,p0_baseline,Baseline,feat,0.459701,0.771930,0.576239,798
2,strict_taxonomy,p0_baseline,Baseline,build,0.250000,0.054545,0.089552,330
3,strict_taxonomy,p0_baseline,Baseline,ci,0.414343,0.412698,0.413519,252
4,strict_taxonomy,p0_baseline,Baseline,docs,0.802390,0.781558,0.791837,1117
5,strict_taxonomy,p0_baseline,Baseline,style,0.044944,0.571429,0.083333,7
6,strict_taxonomy,p0_baseline,Baseline,refactor,0.284644,0.730769,0.409704,416
7,strict_taxonomy,p0_baseline,Baseline,perf,0.600000,0.300000,0.400000,100
8,strict_taxonomy,p0_baseline,Baseline,test,0.603286,0.717877,0.655612,358
9,strict_taxonomy,p0_baseline,Baseline,chore,0.624726,0.549038,0.584442,2080


## 3. Validate or create the class-level metrics CSV

If `class_metrics_baseline_vs_code_disambiguation.csv` already exists, the notebook verifies its schema, row keys, supports, and metric values against the predictions. This prevents a stale CSV from silently diverging from the published prediction artifact.

If it does not exist, the recomputed table is written to `results/rq3_2/metrics/`.


In [4]:
KEY_COLUMNS = [
    "policy",
    "prompt",
    "prompt_label",
    "class",
]
VALUE_COLUMNS = [
    "precision",
    "recall",
    "f1",
    "support",
]

if METRICS_PATH.is_file():
    published_metrics = pd.read_csv(METRICS_PATH)

    expected_columns = KEY_COLUMNS + VALUE_COLUMNS
    if list(published_metrics.columns) != expected_columns:
        raise ValueError(
            "Unexpected metrics CSV schema.\n"
            f"Expected: {expected_columns}\n"
            f"Found:    {list(published_metrics.columns)}"
        )

    expected_sorted = (
        computed_metrics
        .sort_values(KEY_COLUMNS)
        .reset_index(drop=True)
    )
    published_sorted = (
        published_metrics
        .sort_values(KEY_COLUMNS)
        .reset_index(drop=True)
    )

    if published_sorted[KEY_COLUMNS].astype(str).equals(
        expected_sorted[KEY_COLUMNS].astype(str)
    ) is False:
        raise ValueError(
            "The metrics CSV contains different row keys from the "
            "metrics recomputed from predictions."
        )

    if not np.array_equal(
        published_sorted["support"].to_numpy(dtype=int),
        expected_sorted["support"].to_numpy(dtype=int),
    ):
        raise ValueError(
            "Class support in the metrics CSV does not match the predictions."
        )

    for metric in ["precision", "recall", "f1"]:
        if not np.allclose(
            published_sorted[metric].to_numpy(dtype=float),
            expected_sorted[metric].to_numpy(dtype=float),
            rtol=0,
            atol=1e-12,
            equal_nan=True,
        ):
            max_difference = np.max(
                np.abs(
                    published_sorted[metric].to_numpy(dtype=float)
                    - expected_sorted[metric].to_numpy(dtype=float)
                )
            )
            raise ValueError(
                f"{metric}: published CSV differs from recomputed values. "
                f"Maximum absolute difference = {max_difference:.3e}"
            )

    print("Published class-metrics CSV matches predictions exactly.")

else:
    METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
    computed_metrics.to_csv(
        METRICS_PATH,
        index=False,
        float_format="%.15g",
    )
    print(f"Created: {METRICS_PATH}")


Published class-metrics CSV matches predictions exactly.


## 4. Diagnostic comparison used in the paper

The table below places baseline F1, final-prompt F1, \(\Delta F_1\), and support side by side. Classes are ordered by the F1 change produced by code disambiguation.


In [5]:
class_comparison = (
    baseline_metrics[
        ["class", "precision", "recall", "f1", "support"]
    ]
    .rename(columns={
        "precision": "baseline_precision",
        "recall": "baseline_recall",
        "f1": "baseline_f1",
    })
    .merge(
        final_metrics[
            ["class", "precision", "recall", "f1"]
        ].rename(columns={
            "precision": "code_precision",
            "recall": "code_recall",
            "f1": "code_f1",
        }),
        on="class",
        validate="one_to_one",
    )
)

class_comparison["delta_precision"] = (
    class_comparison["code_precision"]
    - class_comparison["baseline_precision"]
)
class_comparison["delta_recall"] = (
    class_comparison["code_recall"]
    - class_comparison["baseline_recall"]
)
class_comparison["delta_f1"] = (
    class_comparison["code_f1"]
    - class_comparison["baseline_f1"]
)

class_comparison = class_comparison.sort_values(
    "delta_f1",
    ascending=False,
).reset_index(drop=True)

display(
    class_comparison[
        [
            "class",
            "support",
            "baseline_f1",
            "code_f1",
            "delta_f1",
            "baseline_precision",
            "code_precision",
            "delta_precision",
            "baseline_recall",
            "code_recall",
            "delta_recall",
        ]
    ].style.format({
        "baseline_f1": "{:.3f}",
        "code_f1": "{:.3f}",
        "delta_f1": "{:+.3f}",
        "baseline_precision": "{:.3f}",
        "code_precision": "{:.3f}",
        "delta_precision": "{:+.3f}",
        "baseline_recall": "{:.3f}",
        "code_recall": "{:.3f}",
        "delta_recall": "{:+.3f}",
    })
)


,class,support,baseline_f1,code_f1,delta_f1,baseline_precision,code_precision,delta_precision,baseline_recall,code_recall,delta_recall
0,build,330,0.090,0.272,+0.183,0.250,0.188,-0.062,0.055,0.494,+0.439
1,perf,100,0.400,0.494,+0.094,0.600,0.672,+0.072,0.300,0.390,+0.090
2,fix,2104,0.578,0.657,+0.079,0.694,0.752,+0.058,0.495,0.583,+0.088
3,refactor,416,0.410,0.455,+0.045,0.285,0.352,+0.068,0.731,0.642,-0.089
4,ci,252,0.414,0.447,+0.034,0.414,0.382,-0.032,0.413,0.540,+0.127
5,test,358,0.656,0.684,+0.029,0.603,0.620,+0.017,0.718,0.763,+0.045
6,docs,1117,0.792,0.815,+0.024,0.802,0.818,+0.016,0.782,0.813,+0.031
7,style,7,0.083,0.077,-0.006,0.045,0.042,-0.003,0.571,0.429,-0.143
8,feat,798,0.576,0.565,-0.011,0.460,0.453,-0.007,0.772,0.751,-0.021
9,others,375,0.070,0.005,-0.065,0.095,0.050,-0.045,0.056,0.003,-0.053


## 5. Reproduce the class-level F1 figure

The figure mirrors the paper's class-level comparison: circles represent the baseline, diamonds represent code disambiguation, and classes are ordered by \(\Delta F_1\) (ties broken by support). Each class label shows the support and the baseline → code-disambiguation F1 values, and the colored number next to each pair is the exact \(\Delta F_1\).

A vector PDF (for the article) and a high-resolution PNG preview (not for the article) are written to `results/rq3_2/figures/`.


In [ ]:
from matplotlib.lines import Line2D


# ============================================================
# CONFIGURATION
# ============================================================

baseline_label = PROMPT_LABELS[BASELINE_PROMPT]
compare_label = PROMPT_LABELS[FINAL_PROMPT]

LINE_COLOR = "#BDBDBD"

# Colorblind-friendly colors.
BASELINE_COLOR = "#0072B2"
COMPARE_COLOR = "#D55E00"

DELTA_POS_COLOR = "#007A33"
DELTA_NEG_COLOR = "#B2182B"
DELTA_NEU_COLOR = "#222222"

BASELINE_MARKER = "o"
COMPARE_MARKER = "D"


# ============================================================
# PREPARE CLASS-LEVEL DATA
# ============================================================

# Retain only classes represented in the evaluation dataset, and break ties
# in delta_f1 by support so the ordering is fully deterministic.
plot_data = (
    class_comparison
    .loc[class_comparison["support"] > 0]
    .sort_values(
        ["delta_f1", "support"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)


# ============================================================
# TWO-LINE CLASS LABELS
# ============================================================

# Example:
#
# build (n=330)
# 0.090 → 0.272
#
# The displayed F1 values are rounded to three decimals, while delta_f1
# remains computed from full precision in the cell above.
plot_data["label"] = plot_data.apply(
    lambda row: (
        f"{row['class']} "
        f"(n={int(row['support']):,})\n"
        f"{row['baseline_f1']:.3f} → {row['code_f1']:.3f}"
    ),
    axis=1,
)


# ============================================================
# TYPOGRAPHY AND OUTPUT SIZE
# ============================================================

plt.rcParams.update({
    "font.size": 7.5,
    "axes.labelsize": 7.5,
    "xtick.labelsize": 7.0,

    # Slightly smaller because y labels now have two lines.
    "ytick.labelsize": 6.7,

    "legend.fontsize": 7.0,
    "axes.linewidth": 0.7,
    "grid.linewidth": 0.5,

    # Embed TrueType fonts in the PDF instead of Type 3 bitmaps.
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

number_of_classes = len(plot_data)
y_positions = np.arange(number_of_classes)

# Sized for a single-column figure in the paper; slightly taller than a
# naive default because every class label now occupies two lines.
fig, ax = plt.subplots(
    figsize=(3.32, 4.95),
    constrained_layout=True,
)


# ============================================================
# DRAW DUMBBELL PLOT
# ============================================================

for yi, (_, row) in zip(y_positions, plot_data.iterrows()):
    baseline_value = float(row["baseline_f1"])
    compare_value = float(row["code_f1"])
    delta = float(row["delta_f1"])

    # Connecting line.
    ax.plot(
        [baseline_value, compare_value],
        [yi, yi],
        color=LINE_COLOR,
        linewidth=1.2,
        solid_capstyle="round",
        zorder=1,
    )

    # Baseline marker.
    ax.scatter(
        baseline_value,
        yi,
        marker=BASELINE_MARKER,
        s=20,
        facecolor=BASELINE_COLOR,
        edgecolor=BASELINE_COLOR,
        linewidth=0.6,
        zorder=3,
    )

    # Code-disambiguation marker.
    ax.scatter(
        compare_value,
        yi,
        marker=COMPARE_MARKER,
        s=23,
        facecolor=COMPARE_COLOR,
        edgecolor=COMPARE_COLOR,
        linewidth=0.6,
        zorder=3,
    )

    if delta > 0:
        delta_color = DELTA_POS_COLOR
    elif delta < 0:
        delta_color = DELTA_NEG_COLOR
    else:
        delta_color = DELTA_NEU_COLOR

    # Place the delta label shortly after the rightmost marker, flipping to
    # the left of the leftmost marker if that would run off the axis.
    text_x = max(baseline_value, compare_value) + 0.025
    text_alignment = "left"

    if text_x > 0.985:
        text_x = min(baseline_value, compare_value) - 0.025
        text_alignment = "right"

    ax.text(
        text_x,
        yi,
        f"{delta:+.3f}",
        va="center",
        ha=text_alignment,
        fontsize=7.0,
        color=delta_color,
        clip_on=True,
        zorder=4,
    )


# ============================================================
# AXES
# ============================================================

ax.set_yticks(y_positions)
ax.set_yticklabels(plot_data["label"])

# Improve spacing between the two lines of each y-axis label.
for label in ax.get_yticklabels():
    label.set_linespacing(1.10)

# Largest positive delta at the top.
ax.invert_yaxis()

ax.set_xlabel("F1-score")
ax.set_ylabel(None)

# Preserve the complete and interpretable F1 range.
ax.set_xlim(-0.015, 1.015)
ax.set_xticks(np.arange(0.0, 1.01, 0.2))

ax.grid(axis="x", alpha=0.25)
ax.grid(axis="y", visible=False)
ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


# ============================================================
# LEGEND
# ============================================================

legend_handles = [
    Line2D(
        [0], [0],
        marker=BASELINE_MARKER,
        linestyle="None",
        markerfacecolor=BASELINE_COLOR,
        markeredgecolor=BASELINE_COLOR,
        markersize=4.5,
        label=baseline_label,
    ),
    Line2D(
        [0], [0],
        marker=COMPARE_MARKER,
        linestyle="None",
        markerfacecolor=COMPARE_COLOR,
        markeredgecolor=COMPARE_COLOR,
        markersize=4.5,
        label=compare_label,
    ),
]

ax.legend(
    handles=legend_handles,
    frameon=False,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.01),
    ncol=2,
    columnspacing=1.2,
    handletextpad=0.4,
    borderaxespad=0,
)


# ============================================================
# EXPORT
# ============================================================

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Vector version for the article.
fig.savefig(
    FIGURE_PDF_PATH,
    format="pdf",
    facecolor="white",
)

# Raster preview at high resolution; do not use this file in the article.
fig.savefig(
    FIGURE_PATH,
    dpi=600,
    facecolor="white",
)

plt.show()

print(f"Vector figure saved to: {FIGURE_PDF_PATH}")
print(f"Preview saved to: {FIGURE_PATH}")


## 6. Data-driven summary

This cell reports the main descriptive changes without hard-coding the values used in the article.


In [7]:
largest_gain = class_comparison.loc[
    class_comparison["delta_f1"].idxmax()
]
largest_decline = class_comparison.loc[
    class_comparison["delta_f1"].idxmin()
]

print(
    "Largest F1 improvement:"
    f" {largest_gain['class']} "
    f"{largest_gain['baseline_f1']:.3f} -> "
    f"{largest_gain['code_f1']:.3f} "
    f"(delta={largest_gain['delta_f1']:+.3f})"
)

print(
    "Largest F1 decline:"
    f" {largest_decline['class']} "
    f"{largest_decline['baseline_f1']:.3f} -> "
    f"{largest_decline['code_f1']:.3f} "
    f"(delta={largest_decline['delta_f1']:+.3f})"
)

print("\nOrdered F1 changes:")
for _, row in class_comparison.iterrows():
    print(
        f"  {row['class']:<8} "
        f"{row['baseline_f1']:.3f} -> "
        f"{row['code_f1']:.3f} "
        f"({row['delta_f1']:+.3f}), "
        f"support={int(row['support'])}"
    )


Largest F1 improvement: build 0.090 -> 0.272 (delta=+0.183)
Largest F1 decline: chore 0.584 -> 0.426 (delta=-0.159)

Ordered F1 changes:
  build    0.090 -> 0.272 (+0.183), support=330
  perf     0.400 -> 0.494 (+0.094), support=100
  fix      0.578 -> 0.657 (+0.079), support=2104
  refactor 0.410 -> 0.455 (+0.045), support=416
  ci       0.414 -> 0.447 (+0.034), support=252
  test     0.656 -> 0.684 (+0.029), support=358
  docs     0.792 -> 0.815 (+0.024), support=1117
  style    0.083 -> 0.077 (-0.006), support=7
  feat     0.576 -> 0.565 (-0.011), support=798
  others   0.070 -> 0.005 (-0.065), support=375
  chore    0.584 -> 0.426 (-0.159), support=2080


## Reproducibility notes

- This is a **descriptive pooled class-level analysis**, not a second inferential analysis.
- The project-level Macro-F1 analyses remain in `rq3_2_prompt_refinement.ipynb`.
- `others` is treated as a valid semantic class only when explicitly predicted.
- Strict invalid outputs and final abstentions remain incorrect predictions but are not introduced as semantic classes.
- The class-metrics CSV is derived from `predictions.csv.gz` and can be validated or regenerated by this notebook.
- The figure is generated from the same recomputed metrics, keeping the table, text, and visualization tied to the published prediction artifact.
